# Data Balancing using SMOTENC

In [ ]:


# =============================================================================
# 1. Import Required Libraries
# =============================================================================

import pandas as pd

from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTENC


# =============================================================================
# 2. Define File Paths
# =============================================================================

DATA_FILE = r"C:\Users\HPC\Downloads\HPAI_Climate_data.csv"

OUTPUT_FILE = r"C:\Users\HPC\Downloads\balanced_data.csv"


# =============================================================================
# 3. Load Dataset
# =============================================================================

df = pd.read_csv(DATA_FILE)

# Remove leading/trailing spaces from column names
df.columns = df.columns.str.strip()

print("Dataset Shape:", df.shape)
print(df.head())


# =============================================================================
# 4. Rename Climate Variables
# =============================================================================

column_mapping = {
    "UVA_Irradiance (kW-hr/m^2/day)": "UVA_Irradiance",
    "UVB_Irradiance (kW-hr/m^2/day)": "UVB_Irradiance",
    "UV_Index (W m-2 x 40)": "UV_Index",
    "Wind_Speed at 2 Meters (m/s)": "Wind_2M",
    "Temperature at 2 Meters (C)": "Temp",
    "Temperature at 2 Meters Maximum (C)": "Temp_Max",
    "Temperature at 2 Meters Minimum (C)": "Temp_Min",
    "Specific Humidity at 2 Meters (g/kg)": "Spec_Humidity",
    "Relative Humidity at 2 Meters (%)": "Rel_Humidity",
    "Precipitation Corrected (mm/day)": "Precipitation",
    "Surface Pressure (kPa)": "Pressure",
    "Wind Speed at 10 Meters (m/s)": "Wind_10M",
    "Wind Direction at 10 Meters (Degrees)": "Wind_Dir_10M"
}

df.rename(columns=column_mapping, inplace=True)


# =============================================================================
# 5. Define Features and Target
# =============================================================================

TARGET = "Status"

NUMERIC_FEATURES = [
    "UVA_Irradiance",
    "UVB_Irradiance",
    "UV_Index",
    "Wind_2M",
    "Temp",
    "Temp_Max",
    "Temp_Min",
    "Spec_Humidity",
    "Rel_Humidity",
    "Precipitation",
    "Pressure",
    "Wind_10M",
    "Wind_Dir_10M",
    "Year"
]

CATEGORICAL_FEATURES = [
    "Month",
    "States"
]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET].copy()

print("\nOriginal Class Distribution:")
print(Counter(y))


# =============================================================================
# 6. Handle Missing Values
# =============================================================================

numeric_imputer = SimpleImputer(strategy="median")

X[NUMERIC_FEATURES] = numeric_imputer.fit_transform(
    X[NUMERIC_FEATURES]
)


# =============================================================================
# 7. Encode Categorical Variables
# =============================================================================

label_encoders = {}

for column in CATEGORICAL_FEATURES:

    encoder = LabelEncoder()

    X[column] = encoder.fit_transform(X[column])

    label_encoders[column] = encoder


# =============================================================================
# 8. Identify Categorical Columns for SMOTENC
# =============================================================================

categorical_indices = [
    X.columns.get_loc(col)
    for col in CATEGORICAL_FEATURES
]

print("\nCategorical Feature Indices:")
print(categorical_indices)


# =============================================================================
# 9. Define Sampling Strategy
# =============================================================================

class_counts = Counter(y)

majority_class_count = class_counts["Outbreak"]

target_minority_count = int(majority_class_count * 0.70)

sampling_strategy = {
    "Non_Outbreak": target_minority_count
}

print("\nSampling Strategy:")
print(sampling_strategy)


# =============================================================================
# 10. Apply SMOTENC
# =============================================================================

smote_nc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy=sampling_strategy,
    k_neighbors=5,
    random_state=42
)

X_resampled, y_resampled = smote_nc.fit_resample(X, y)

print("\nClass Distribution After SMOTENC:")
print(Counter(y_resampled))


# =============================================================================
# 11. Decode Categorical Variables
# =============================================================================

for column, encoder in label_encoders.items():

    X_resampled[column] = encoder.inverse_transform(
        X_resampled[column]
        .round()
        .astype(int)
    )


# =============================================================================
# 12. Create Balanced Dataset
# =============================================================================

balanced_df = X_resampled.copy()

balanced_df[TARGET] = y_resampled

print("\nBalanced Dataset Shape:")
print(balanced_df.shape)

print(balanced_df.head())


# =============================================================================
# 13. Save Balanced Dataset
# =============================================================================

balanced_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"\nBalanced dataset saved successfully:")
print(OUTPUT_FILE)